# TinyBERT-XAI Dataset and Model Explorer

This notebook inspects the TweetEval sentiment dataset, the tokenized pilot batch, and the teacher/student model outputs used by the iteration 0 smoke test.

In [2]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset

from tinybert_xai import KDPair

pair = KDPair.for_dataset("tweet_eval/sentiment")
device = pair.device

print(f"device: {device}")
print(f"dataset: {pair.spec.hf_path} [{pair.spec.hf_config}]")
print(f"teacher: {pair.teacher_checkpoint}")
print(f"student: {pair.student_checkpoint}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device: cuda
dataset: cardiffnlp/tweet_eval [sentiment]
teacher: bert-base-uncased
student: huawei-noah/TinyBERT_General_4L_312D


## Load the Dataset Splits

In [3]:
splits = load_dataset(pair.spec.hf_path, pair.spec.hf_config)
split_sizes = {name: split.num_rows for name, split in splits.items()}
split_sizes

{'train': 45615, 'test': 12284, 'validation': 2000}

In [4]:
train_ds = load_dataset(pair.spec.hf_path, pair.spec.hf_config, split="train")
assert isinstance(train_ds, Dataset)

print(train_ds)
print("columns:", train_ds.column_names)
print("features:", train_ds.features)

Dataset({
    features: ['text', 'label'],
    num_rows: 45615
})
columns: ['text', 'label']
features: {'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['negative', 'neutral', 'positive'], id=None)}


## Inspect Raw Examples

In [ ]:
label_names = pair.label_names
label_names

In [5]:
raw_preview = train_ds.select(range(10)).to_pandas()
raw_preview["label_name"] = raw_preview["label"].map(lambda label_id: label_names[label_id])
raw_preview[["label", "label_name", "text"]]

,label,label_name,text
0,2,positive,"""QT @user In the original draft of the 7th boo..."
1,1,neutral,"""Ben Smith / Smith (concussion) remains out of..."
2,1,neutral,Sorry bout the stream last night I crashed out...
3,1,neutral,Chase Headley's RBI double in the 8th inning o...
4,2,positive,@user Alciato: Bee will invest 150 million in ...
5,2,positive,@user LIT MY MUM 'Kerry the louboutins I wonde...
6,2,positive,"""\"""""""" SOUL TRAIN\"""""""" OCT 27 HALLOWEEN SPECIA..."
7,0,negative,So disappointed in wwe summerslam! I want to s...
8,2,positive,"""This is the last Sunday w/o football .....,NF..."
9,1,neutral,@user @user CENA & AJ sitting in a tree K-I-S-...


In [6]:
label_counts = pd.Series(train_ds["label"]).value_counts().sort_index()
pd.DataFrame(
    {
        "label": label_counts.index,
        "label_name": [label_names[label_id] for label_id in label_counts.index],
        "count": label_counts.values,
    }
)

,label,label_name,count
0,0,negative,7093
1,1,neutral,20673
2,2,positive,17849


## Load Teacher and Student Models

In [ ]:
def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

pd.DataFrame([
    {"model": "teacher", "name": pair.teacher_checkpoint, "params_m": count_params(pair.teacher) / 1e6},
    {"model": "student", "name": pair.student_checkpoint, "params_m": count_params(pair.student) / 1e6},
])

## Inspect the Tokenized Pilot Batch

In [ ]:
batch = pair.sample_batch(n=4)
{key: tuple(value.shape) for key, value in batch.items()}

In [ ]:
decoded = pair.tokenizer.batch_decode(batch["input_ids"], skip_special_tokens=True)
pd.DataFrame(
    {
        "label": batch["labels"].tolist(),
        "label_name": [label_names[i] for i in batch["labels"].tolist()],
        "decoded_text": decoded,
    }
)

## Forward Pass and Output Shapes

In [ ]:
out = pair.forward(batch)

shape_rows = [
    {"model": "teacher", "tensor": "logits",           "shape": tuple(out.teacher.logits.shape)},
    {"model": "teacher", "tensor": "hidden_states[0]", "shape": tuple(out.teacher.hidden_states[0].shape)},
    {"model": "teacher", "tensor": "hidden_states[-1]","shape": tuple(out.teacher.hidden_states[-1].shape)},
    {"model": "teacher", "tensor": "attentions[0]",    "shape": tuple(out.teacher.attentions[0].shape)},
    {"model": "student", "tensor": "logits",           "shape": tuple(out.student.logits.shape)},
    {"model": "student", "tensor": "hidden_states[0]", "shape": tuple(out.student.hidden_states[0].shape)},
    {"model": "student", "tensor": "hidden_states[-1]","shape": tuple(out.student.hidden_states[-1].shape)},
    {"model": "student", "tensor": "attentions[0]",    "shape": tuple(out.student.attentions[0].shape)},
]

pd.DataFrame(shape_rows)

In [ ]:
teacher_probs = torch.softmax(out.teacher.logits.detach().cpu(), dim=-1)
student_probs = torch.softmax(out.student.logits.detach().cpu(), dim=-1)

pd.DataFrame(
    {
        "text": decoded,
        "gold_label":   [label_names[i] for i in batch["labels"].tolist()],
        "teacher_pred": [label_names[i] for i in teacher_probs.argmax(dim=-1).tolist()],
        "student_pred": [label_names[i] for i in student_probs.argmax(dim=-1).tolist()],
        "teacher_probs": teacher_probs.tolist(),
        "student_probs": student_probs.tolist(),
    }
)